# NLP and Naive Bayes

Text classification and sentiment analysis for blog posts.


## Assignment Explanation

### Objective
The objective of this assignment is to classify blog posts into categories using Naive Bayes and perform sentiment analysis on the text data.

### Methodology
The blog dataset is loaded and the text and category columns are selected. Text preprocessing is performed by converting text to lowercase, removing links, removing symbols, and cleaning extra spaces. TF-IDF is used to convert text into numerical features. A Multinomial Naive Bayes classifier is trained to predict blog categories.

Sentiment analysis is performed using polarity scores. Posts are classified as positive, negative, or neutral based on their sentiment polarity.

### Interpretation
Naive Bayes is suitable for text classification because it works well with word-frequency features. Sentiment analysis gives additional insight into the emotional tone of blog posts across categories.


In [4]:
import warnings
warnings.filterwarnings('ignore')
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix



df = pd.read_csv(r'C:\Users\lenovo\Downloads\Assignments\NLP and Naive Bayes\blogs.csv')
df.head()


,Data,Labels
0,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,alt.atheism
1,Newsgroups: alt.atheism\nPath: cantaloupe.srv....,alt.atheism
2,Path: cantaloupe.srv.cs.cmu.edu!das-news.harva...,alt.atheism
3,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,alt.atheism
4,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:53...,alt.atheism


In [5]:
text_col = 'Data' if 'Data' in df.columns else df.select_dtypes(include='object').columns[0]
label_col = 'Labels' if 'Labels' in df.columns else df.select_dtypes(include='object').columns[-1]
df = df[[text_col, label_col]].dropna()
df[label_col].value_counts().head(10)


Labels
alt.atheism               100
comp.graphics             100
talk.politics.misc        100
talk.politics.mideast     100
talk.politics.guns        100
soc.religion.christian    100
sci.space                 100
sci.med                   100
sci.electronics           100
sci.crypt                 100
Name: count, dtype: int64

In [6]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\\S+|www\\S+', ' ', text)
    text = re.sub(r'[^a-z\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

df['clean_text'] = df[text_col].apply(clean_text)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df[label_col], test_size=0.2, random_state=42, stratify=df[label_col])
model = Pipeline([('tfidf', TfidfVectorizer(stop_words='english', max_features=20000)), ('nb', MultinomialNB())])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred))


                          precision    recall  f1-score   support

             alt.atheism       0.75      0.60      0.67        20
           comp.graphics       0.77      0.85      0.81        20
 comp.os.ms-windows.misc       0.83      1.00      0.91        20
comp.sys.ibm.pc.hardware       0.72      0.90      0.80        20
   comp.sys.mac.hardware       0.94      0.75      0.83        20
          comp.windows.x       0.92      0.60      0.73        20
            misc.forsale       0.80      0.80      0.80        20
               rec.autos       0.86      0.95      0.90        20
         rec.motorcycles       0.95      0.90      0.92        20
      rec.sport.baseball       0.89      0.85      0.87        20
        rec.sport.hockey       0.87      1.00      0.93        20
               sci.crypt       0.95      1.00      0.98        20
         sci.electronics       0.87      0.65      0.74        20
                 sci.med       0.94      0.80      0.86        20
         

In [9]:
from textblob import TextBlob

def sentiment_label(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0.05:
        return 'positive'
    if polarity < -0.05:
        return 'negative'
    return 'neutral'

def sentiment_label(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0.05:
        return 'positive'
    if polarity < -0.05:
        return 'negative'
    return 'neutral'

df['sentiment'] = df['clean_text'].head(1000).apply(sentiment_label)
df['sentiment'].value_counts()
df['sentiment'].value_counts()


sentiment
positive    580
neutral     318
negative    102
Name: count, dtype: int64

In [10]:
pd.crosstab(df.loc[df['sentiment'].notna(), label_col], df.loc[df['sentiment'].notna(), 'sentiment']).head(20)


sentiment,negative,neutral,positive
Labels,,,
alt.atheism,12,35,53
comp.graphics,10,34,56
comp.os.ms-windows.misc,11,24,65
comp.sys.ibm.pc.hardware,10,29,61
comp.sys.mac.hardware,6,37,57
comp.windows.x,8,41,51
misc.forsale,12,21,67
rec.autos,10,27,63
rec.motorcycles,10,38,52
